# Extensive Workflow Demo

In [ ]:
import numpy as np
import pandas as pd
from numba import njit, prange # for code speedup

import superstats as sup

In Superstats, a dynamic genreative model is built from two pieces: a **low-level
observation model** (e.g., a cognitive model such as the Diffusion Decision Model) that generates data at each time step, and a **high-level transition model** that describes how the model's parameters evolve over time.

A typical amortized Bayesian workflow ([Li et al., 2026](https://openreview.net/forum?id=osV7adJlKD)) consists of the following steps:

1. **Define the observation model** as a data simulator.
2. **Specify a joint prior**, assigning a transition model to each parameter that
   should vary over time, and a standard prior to those that should not.
3. **Prior push forward checks:** Simulate from the generative model and ask whether
   the implied parameter trajectories and data are consistent with your domain
   knowledge. Adjust the priors and transition models until they are.
4. **Set up the amortized Bayesian workflow:** specify a neural approximator consisting of a summary and inference network.
5. **Train the neural approximator** on simulations from the generative model.
6. **Model verification:** Check that the approximate posteriors are well calibrated (via simulation-based calibration) and that the model and design can answer your question at all (via parameter recovery and posterior contraction). If they cannot, return to steps 1. – 2. and revise.
7. **Fit empirical data** for any number of datasets, at negligible cost.
8. **Evaluate the absolute model fit** Re-simulate data from the posterior and ask
   whether the model reproduces the patterns you care about. A model that misses
   them is not worth interpreting, no matter how well it did in step 6.
9. **Inspect the posteriors** of the time-varying and time-invariant parameters.

Superstats provides tools with sensible defaults for each step, while
remaining fully customizable wherever you need it.

## Define the Observation Model

The observation model $\mathcal{G}$ is the low-level simulator: the generative model whose parameters are either allowed to vary over time. Formally, it implements

$$x_t = \mathcal{G}(\theta_t, z_t), \qquad t = 1, \dots, T$$

where $\theta_t$ collects the model parameters at time step $t$ and $z_t$ is the simulator's own source of randomness (e.g. diffusion noise). `superstats` requires only that $\mathcal{G}$ can be
*simulated* — no closed-form likelihood is needed.

**Function signature.** `superstats` expects a plain Python (or `numba`-jitted) function that:

- takes each model parameter as a keyword argument, passed as an array of shape `(num_steps,)` one value per time step. Time-invariant parameters are tiled internally to `num_steps` before the call, so every parameter arrives with the same shape regardless of whether it was declared as time-varying, time-invariant, or fixed;
- returns a dict mapping observation names to arrays of shape `(num_steps,)`, i.e. one named observed variable per time step.

```python
def observation_model(
    param_1: np.ndarray,   # shape (num_steps,)
    param_2: np.ndarray,   # shape (num_steps,)
    ...
) -> dict[str, np.ndarray]: # each value has shape (num_steps,)
    ...
```

### Example: Diffusion Decision Model (DDM)

The diffusion decision model (DDM; [Ratcliff, 1978](https://doi.org/10.1037/0033-295X.85.2.59)) describes binary decisions as noisy evidence accumulation toward one of two boundaries:

$$dx = v_t \, dt + \sigma \, dW_t.$$

Evidence $x$ starts at $\text{bias}_t \cdot a_t$ and accumulates until it hits $a_t$ (upper boundary,
choice $=1$) or $0$ (lower boundary, choice $=0$); the response time is $\tau_t$ (non-decision time)
plus the time to reach a boundary. $v_t$ is the drift rate, $a_t$ the boundary separation
(speed–accuracy trade-off), and $\text{bias}_t \in (0,1)$ the relative starting point — $0.5$ is
unbiased, values above or below shift the start toward the upper or lower boundary, respectively.

`sample_ddm` integrates this via Euler–Maruyama: at each step of size `dt`, it adds drift `v_t * dt` and Gaussian noise scaled by `sigma * sqrt(dt)`, checking for a boundary crossing. Trials that don't resolve within `max_steps` are marked as timeouts (RT $= -1.0$). Trials are simulated in parallel via `numba`.

In [ ]:
import numpy as np
from numba import njit, prange

@njit(parallel=True, fastmath=True)
def sample_ddm(
    v: np.ndarray,
    a: np.ndarray,
    tau: np.ndarray,
    bias: np.ndarray,
    sigma: float = 1.0,
    dt: float = 0.001,
    max_steps: int = 10000,
) -> dict[str, np.ndarray]:
    num_steps = v.shape[0]
    response_time = np.empty(num_steps, dtype=np.float32)
    choice = np.empty(num_steps, dtype=np.float32)
    noise_scale = sigma * np.sqrt(dt)

    for i in prange(num_steps):
        v_t = v[i]
        a_t = a[i]
        t = tau[i]
        x = bias[i] * a_t
        drift_dt = v_t * dt

        for step in range(max_steps):
            t += dt
            x += drift_dt + noise_scale * np.random.normal()
            if x >= a_t:
                response_time[i] = t
                choice[i] = 1.0
                break
            if x <= 0.0:
                response_time[i] = t
                choice[i] = 0.0
                break
        else:
            response_time[i] = -1.0
            choice[i] = -1.0

    return {"response_time": response_time, "choice": choice}

This specific model and others are already implemented in `superstats`. However, any simulator that respects this contract, one array in per parameter and a dict of named one-dimensional observed variables, can be used.

In [ ]:
model = sample_ddm

# or
model = sup.simulation.cognitive.sample_ddm

## Specify a Joint Prior

Next, we specify for each model parameter a prior distribution or fix it to some value (fixing can either be done directly in the simulator, or at this stage here). `superstats` differntiate between the following parameter types:

- `local_param`: an observation model parameter that is coverned by a transition model and thus time-varying.
- `hyper_param`: a time-invariant transition model parameter.
- `shared_param`: a time-invariant observation model parameter.
- `fixed_param`: an model parameter that is fixed.

All parameter types except `fixed_param` are jointly estimated.

To specify all model parameters we use the `JointPrior` class. For each observation model we either choose to let it be coverned by a transition model and thus be time-varying, get a regular prior and thus be a time-invariant parameter shared across time steps for get not prior and thus be fixed to some value.

### Transition models

One could argue its best to be as agnostic as possible and just use the most flexible transition model for all parameters and just let the data descide what is most plausible. However, we recommend as also in regular Bayesian inference, to regard the transition models as priors on parameters and being not to liberal might be a good thing due to normalization. Also, when we use a very flexible transiiton model then it might also get difficult to get precise estimates of some parameters as they start to trade-off with other ones. Therefore, we suggest that users go over each observation model parameter and think whether there are reasons why we expect paramteres changes over time. If yes, how do we expect these changes to look like?

`superstats` comes with a several different transition models:

- `RandomWalk`: Gaussian random walk with `sigma` that coverns the standard deviation of the noise. Optionally, one can also estimate `delta` as an additive drift.
- `AutoRegression`: Implements AR1 and is the same as RandomWalk except it allows for an additional hyper parameter `phi` that governs the strength of the mean reverting behavior.
- `OrnsteinUhlenbeck`: ...
- `Jump`: Allows for sudden shits based on samples from a proposal distribution and estimation of shift frequency `p_jump`.
- `Mixture`: Allows to combine all of the transitions above and estimate mixture probablities between them. This is particullarly intersting with the Jump transition (e.g., combining `RandomWalk` but also allowing for sudden shifts via `Jump`).
- `GaussianProcess`: Allows estimation of smooth transitions. Different kernels are implemented and can also be combinded additively or multiplicatively.



### Example 

Let us specify a DDM where we only estimate parameter trajectories for the drfit rate $v$ and the threshold $a$.
We assume that the non-decision time $\tau$ does not change over time we, however, still estimate it shared across all time steps, and we fix the starting point to $0.5$ representing no a priori choice biases.
For the drift rate, we assume a simple but flexible random walk. We also want to estimate $\delta$ to get a direct estimate, if the drift rate systematically increases or decreases over time.
For the threshold, we assume a mixture between a random walk, this time without additional drift parameter $\delta$ and Jump transition.

Whenever we specify transition model, we need to specify (or leave it to defaults) the following:
- `bounds`: specify bounds for parameters. Under the hood, parameter trajecotries are generated in an unconstraint space but then transformed via a scaled sigmoid function to remain within the defined bounds.
- `initial_prior`: define a prior to sample an initial parameter value. Keep in mind that, the values are transformed with the scaled sigmoid function.
- `hyper_param`: Set priors for the transition model specific hyperparameters if zou want to estimate them or fix them to some value.

In [ ]:
joint_prior = sup.prior.JointPrior(
    v = sup.transition.RandomWalk(
        bounds=(0, 6),
        initial_prior=sup.prior.Prior(dist="normal", loc=-1, scale=1),
        sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
        delta=sup.prior.Prior(dist="normal", loc=0.0, scale=0.01)
    ),
    a = sup.transition.Mixture(
        bounds=(0.5, 5),
        initial_prior=sup.prior.Prior(dist="normal", loc=-0.5, scale=1.0),
        transitions=[
            sup.transition.RandomWalk(
                sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
                delta=0
            ),
            sup.transition.Jump(
                proposal_prior=sup.prior.Prior(dist="logistic", loc=0, scale=1),
                p_jump=1.0
            )
        ],
        mixture_weights=sup.prior.Prior(dist="dirichlet", alpha=[40.0, 2.0])
    ),
    tau = sup.prior.Prior(dist="halfnormal", scale=0.5),
    bias = 0.5 
)

A few notes concerning the `Mixture` transition.
`bounds` and `initial_prior` needs to be defined once at initialization and is forbidden to also specify, within the actual transitions of the mixture.
When a `Jump` transition is used in the `Mixture` then `p_jump` is automatically fixed to $1.0$ because the `mixture_weights` now coverns the probability of a jump occuring.
It won't be sensible if the Jump transition is selected at a given time step and then there is a chance that no jump actually is happening due to `p_jump < 1.0`.
We already mentioned that after a parameter trajectory is generated in the unconstraint space it is transformed via sclaed sigmoid to ensure each parameter remains within its bounds. 
This means that `proposal_prior=sup.prior.Prior(dist="logistic", loc=0, scale=1)` leads to a uniform distribution between the bounds and means that when a jump occurs the parameter can take on any value within its bounds with equal probabliity.

### Inspect the Priors

#### Time-varying Parameters

In [ ]:
fig = joint_prior.plot_time_varying_prior(
    num_steps=200,
    num_trajectories=5
)

#### Time-invariant Parameters

In [ ]:
fig = joint_prior.plot_time_invariant_prior(
    num_draws=2000
)

#### Both

In [ ]:
fig = joint_prior.plot_joint_prior(
    num_steps=200,
    num_trajectories=5,
    num_draws=2000
)

As soon as we think our priors results in plausible parameters values we can combine our observation model with the joint prior to a generative model via the `GenerativeModel` wrapper. This wrapper manages model simulation and also comes with a handy function to inspect our model's implications.

In [ ]:
generative_model = sup.simulation.GenerativeModel(
    prior=joint_prior,
    model=model
)

Lets simulate some data...

In [ ]:
sim_data = generative_model.sample(
    batch_size=10,
    num_steps=200
)

This returns a dict with 10 prior draws from the joint prior and 10 corresponding datasets, each with 200 time steps.

In [ ]:
print(
    "Dict keys: ", sim_data.keys(), "\n",
    "Shape of response_time: ", sim_data["response_time"].shape, "\n",
    "Shape of choice: ", sim_data["choice"].shape, "\n",
    "Shape of a time-varying param: ", sim_data["v"].shape, "\n",
    "Shape of a time-invariant param: ", sim_data["v_sigma"].shape, "\n",
    sep=""
)

## Prior push forward checks

With `plot_push_forward` we can inspect the implication of our prior spefification on data generated by our observation model. This method allows us to specify several important arguments:

- `num_sims`: how many individual datasets should be simulated
- `num_steps`: how many time steps should each dataset contain
- `data_dim`: which data dimension should be plotted (e.g. in the case of the DDM, do we want to plot response times or choices)
- `kind`: lets us specify if we want to plot distributions or time series / trajectories.
- `aggregation`: Plot seperate datasets if `None` or aggregate across datasets with a function such as mean or median.
- `uncertainty_fun`: If aggreagtion not `None` we can specifiy how we want to plot uncertainty (e.g., standard deviation, 95% credibility interval, etc.)
- ...


### Example: Response Time Distributions for Sparate Datasets

In [ ]:
fig = generative_model.plot_push_forward(
    num_sim=10,
    num_steps=200,
    data_dim="response_time",
    kind="dist",
    aggregation=None,
    num_cols=4
)

### Example: Median Response Time Trajectory Across Datasets

In [ ]:
fig = generative_model.plot_push_forward(
    num_sim=1000,
    num_steps=200,
    data_dim="response_time",
    kind="trajectory",
    aggregation=np.median,
    uncertainty_fun="95hdi",
    marginal=True, # plots marginal distribution next to trajectory
)

### Example: Average Choice Proportions Across Datasets

In [ ]:
fig = generative_model.plot_push_forward(
    num_sim=1000,
    num_steps=200,
    data_dim="choice",
    kind="dist",
    aggregation=np.mean
)

## Set up the amortized Bayesian workflow

In this step, we need to specify the neural approximator, which we than can train to learn to perform Bayesian inference. To this end, we need to specify a summary network, which learns informative summary statistics of synthetic data genereated by our model and an inference network that learns to approximate posterior distributions given those summary statistics.

`superstats` already comes with performand default networks, which leaves users who are not experienced in amortized Bayesian inference with zero effort. Power users can either pass their custom networks or use one of the [networks already implemented in BayesFlow](https://ADD_LINKD).

`superstats` wrappes Bayesflow's BasicWorkflow class, and thus, supports all its functionality. The following lists the most important pieces of a workflow:

- `simulator`: Just our generative model
- `adapter`: Performs transformations of simulator outputs (i.e., data, parameters) before passing them to the networks. Sensible default is used when equal `None`.
- `summary_network`: Neural network that embeddes simulated data. Sensible default is used when equal `None`
- `inference_network`: Neural network for posterior approximation. Sensible default is used when equal `None`
- `checkpoint_filepath`: File path to save trained neural approximator

In [ ]:
workflow = sup.workflow.Workflow(
    simulator=generative_model,
    checkpoint_filepath="checkpoints/demo_ddm"
)

## Train the Neural Approximator

Now we have everything ready to train the neural appoximator. For this step, we have to options. Either we pre-simulate some training and test data and train based on those datasets (i.e., offline training) or we generate syntethic datasets oon the fly during training (i.e., online training).

Usually, offline trainig is much faster and thus should be used during model development to facilitate quick iteration of different model specifiaction. If a model has proven promissing, we can use online training to go reach maximal approximator performance.

### Offline Training

To generate train and test data we can simply call the sample method from the simulator and specify how many datasets (`batch_size`) and how many steps each dataset should contain (`num_steps`).
`superstats` treats all parameters, even time-invariants, during trainig as time series, we, therefore, have to tile them to `num_step` when simulating, which we can achieve with `tile_to_steps=True`.

In [ ]:
train_data = workflow.simulator.sample(
    batch_size=20000,
    num_steps=200,
    tile_to_steps=True
)
test_data = workflow.simulator.sample(
    batch_size=250,
    num_steps=200,
    tile_to_steps=True
)

Let us start training...

In [ ]:
history = workflow.fit_offline(
    data=train_data,
    validation_data=test_data,
    epochs=50,
    batch_size=32
)

After training we can inspect the loss history.
If the average loss is below the validation loss we overfitted and should either use more trainig data or switch to online training (in this case we can never overfit).

In [ ]:
fig = workflow.plot_history(history)

Everything looks healthy, and thus, we can move to model verification

### Online Traning

In [ ]:
history = workflow.fit_online(
    num_steps=200,
    epochs=50,
    num_batches_per_epoch=100,
    batch_size=32
)

In [ ]:
fig = workflow.plot_history(history)

## Model Verification

The next importnat step after training, is to verify our model. We want to make sure that the posterior is well calibrated and we can recovery true data generating parameters as well are able to learn anything from data (i.e., posterior contraction).

To this end, we simulate some data and fit our model to the data. Then we inspect the performance with `superstats` for time-varying and time-invariant separaetz, as time-varying parameters have an additional dimension, namely steps, and we want to make sure that the posterior is well behaved across all or at least most time steps.

In [ ]:
targets = workflow.simulator.sample(
    batch_size=1000,
    num_steps=200,
)

In [ ]:
estimates = workflow.sample(
    data={key: targets[key] for key in workflow.simulator.data_keys},
    num_samples=500 # number of posterior samples
)

### Time-varying Parameters

In [ ]:
fig = workflow.verify_time_varying(
    targets=targets,
    estimates=estimates
)

We look at four metrics (rows) across time steps for each parameter separately (columns):

1. **Pearson Correlation** between true data generating parameter (targets) and posterior medians (estimates).
2. **Normalized Root Mean Squared Error (NRMSE)** between true data generating parameter (targets) and posterior medians (estimates). Normalisation is based on the parameter's prior bootstrap samples. This ensures that the NRMSE ranges between [0, 1] with low values indicating good performance.
3. **Posterior Contraction** ranges between [0, 1] with high values indicating good performance.
4. **Calibration Error** ranges between [0, 1] with low values indicating good performance.

We can also inspect the recovery of full trajectories of some example datasets by using the `plot_time_varying_posterior` function, which allows to pass true parameters via the optional argument `targets`.

In [ ]:
# randomly select 10 from our 1000 simulated datasets
idx = np.random.choice(range(1000), size=10, replace=False)

# subset targets and estimates
targets_subset = ...
estimates_subset = ...

For now we use the function with its default arguments. Later when we focus on the inspection of posteriors, we will look at some of the arguments more cloesly.

In [ ]:
fig = workflow.plot_time_varying_posterior(
    estimates=estimates_subset,
    targets=targets_subset
)

Alternatively, we can also inspect recovery and calibartion at specific time steps by using the `plot_recovery()` and  `plot_calibration()` functions from the `diagnostics` module.
Both functions are simple wrappers of Bayesflow's corresponding diagnositc plot functions.
When we use the diagnostics plotting function through the workflows methods then this approach already does some work for us. E.g. it finds out which parameter actually is time-varying and which not. When we call the functions outside of the method we need to specify this by ourself. Luckily, our simulator has this information as an attribute.

In [ ]:
local_param_keys = workflow.simulator.local_keys

In [ ]:
time_step = 50

# subset targets and estimates
targets_subset = ...
estimates_subset = ...

In [ ]:
fig = sup.diagnostics.plots.plot_recovery(
    estimates=estimates_subset,
    targets=targets_subset,
    variable_keys=local_param_keys
)

In [ ]:
fig = sup.diagnostics.plots.plot_calibration(
    estimates=estimates_subset,
    targets=targets_subset,
    variable_keys=local_param_keys
)

### Time-invariant Parameters

To verify the time-invariant parameters we use the same two functions we just used. Howwever, we can call both of them at the same time as an workflow method called `verify_invariant()`, which returns a recovery and a calibration plot.

In [ ]:
fig_recovery, fig_calibration = workflow.verify_time_invariant(
    estimates=estimates,
    targets=targets
)

## Fit Empirical Data

First, we need to read some data and bring it in the same shape as the simulated data, which means a dict mapping observation names to arrays of shape `(num_datasets, num_steps)`.

In [ ]:
df = pd.read_csv("data/data_color_discrimination.csv")
empiric_data = sup.utils.df_to_array(df, id_col="id", data_cols=("rt", "correct"))
empiric_data.shape

In [ ]:
samples = workflow.sample(
    data=empiric_data,
    num_samples=500
)